In [2]:
!pip install kaggle
import os
os.environ["KAGGLE_API_TOKEN"] = "KGAT_a335547efebd42bd871a1efadb74b59c"
import kaggle
!kaggle datasets download ankitbansal06/retail-orders -f orders.csv


Dataset URL: https://www.kaggle.com/datasets/ankitbansal06/retail-orders
License(s): CC0-1.0
orders.csv.zip: Skipping, found more recently modified local copy (use --force to force download)


In [3]:
import zipfile
zip_ref =zipfile.ZipFile('orders.csv.zip')
zip_ref.extractall()
zip_ref.close() 


In [4]:
#read data from the csv file and process null values
import pandas as pd 
df = pd.read_csv('orders.csv', na_values= ['Not Available', 'unknown'])
df['Ship Mode'].unique()

array(['Second Class', 'Standard Class', nan, 'First Class', 'Same Day'],
      dtype=object)

In [5]:
#rename columns names (make them lower case and add undescore)
#df.rename(collumns= {'Order Id': 'order_id', 'City': 'city'})
df.columns= df.columns.str.lower()
df.columns= df.columns.str.replace(' ', '_')
df.columns
df.head(5)

,order_id,order_date,ship_mode,segment,country,city,state,postal_code,region,category,sub_category,product_id,cost_price,list_price,quantity,discount_percent
0,1,2023-03-01,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Bookcases,FUR-BO-10001798,240,260,2,2
1,2,2023-08-15,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Chairs,FUR-CH-10000454,600,730,3,3
2,3,2023-01-10,Second Class,Corporate,United States,Los Angeles,California,90036,West,Office Supplies,Labels,OFF-LA-10000240,10,10,2,5
3,4,2022-06-18,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Furniture,Tables,FUR-TA-10000577,780,960,5,2
4,5,2022-07-13,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Office Supplies,Storage,OFF-ST-10000760,20,20,2,5


In [6]:
#calculate new columns discount, sale price, and profit 
df['discount']=df['list_price']*df['discount_percent']*0.1
df['sale_price']= df['list_price']- df['discount']
df['profit']= df['sale_price']-df['cost_price']
df

,order_id,order_date,ship_mode,segment,country,city,state,postal_code,region,category,sub_category,product_id,cost_price,list_price,quantity,discount_percent,discount,sale_price,profit
0,1,2023-03-01,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Bookcases,FUR-BO-10001798,240,260,2,2,52.0,208.0,-32.0
1,2,2023-08-15,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Chairs,FUR-CH-10000454,600,730,3,3,219.0,511.0,-89.0
2,3,2023-01-10,Second Class,Corporate,United States,Los Angeles,California,90036,West,Office Supplies,Labels,OFF-LA-10000240,10,10,2,5,5.0,5.0,-5.0
3,4,2022-06-18,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Furniture,Tables,FUR-TA-10000577,780,960,5,2,192.0,768.0,-12.0
4,5,2022-07-13,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Office Supplies,Storage,OFF-ST-10000760,20,20,2,5,10.0,10.0,-10.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9989,9990,2023-02-18,Second Class,Consumer,United States,Miami,Florida,33180,South,Furniture,Furnishings,FUR-FU-10001889,30,30,3,4,12.0,18.0,-12.0
9990,9991,2023-03-17,Standard Class,Consumer,United States,Costa Mesa,California,92627,West,Furniture,Furnishings,FUR-FU-10000747,70,90,2,4,36.0,54.0,-16.0
9991,9992,2022-08-07,Standard Class,Consumer,United States,Costa Mesa,California,92627,West,Technology,Phones,TEC-PH-10003645,220,260,2,2,52.0,208.0,-12.0
9992,9993,2022-11-19,Standard Class,Consumer,United States,Costa Mesa,California,92627,West,Office Supplies,Paper,OFF-PA-10004041,30,30,4,3,9.0,21.0,-9.0


In [7]:
#conver order date from object data type to datetime
df['order_date']=pd.to_datetime(df['order_date'], format="%Y-%m-%d")
df.dtypes

order_id                     int64
order_date          datetime64[ns]
ship_mode                   object
segment                     object
country                     object
city                        object
state                       object
postal_code                  int64
region                      object
category                    object
sub_category                object
product_id                  object
cost_price                   int64
list_price                   int64
quantity                     int64
discount_percent             int64
discount                   float64
sale_price                 float64
profit                     float64
dtype: object

In [8]:
#drop cost price, list price and discount percent columns
df.drop(columns=['list_price','cost_price', 'discount_percent'], inplace=True)
df.columns


Index(['order_id', 'order_date', 'ship_mode', 'segment', 'country', 'city',
       'state', 'postal_code', 'region', 'category', 'sub_category',
       'product_id', 'quantity', 'discount', 'sale_price', 'profit'],
      dtype='object')

In [9]:
#loading the data into sql server using replace option
#import sqlalchemy as sal 
#engine = sal.create_engine('mssql://DESKTOP-Q1V9UUB\\SQLEXPRESS/master?driver=ODBC+DRIVER+17+FOR+SERVER')
#conn=engine.connect()

import sqlalchemy as sal

engine = sal.create_engine(
    r"mssql+pyodbc://@DESKTOP-Q1V9UUB\SQLEXPRESS/master?driver=ODBC+Driver+17+for+SQL+Server"
)

conn = engine.connect()
print("Connected successfully!")

Connected successfully!


In [19]:
df.to_sql('df_orders', con=conn, index=False, if_exists = 'replace')
df.to_sql('df_orders', con=conn, index=False, if_exists = 'append')
df.columns

Index(['order_id', 'order_date', 'ship_mode', 'segment', 'country', 'city',
       'state', 'postal_code', 'region', 'category', 'sub_category',
       'product_id', 'quantity', 'discount', 'sale_price', 'profit'],
      dtype='object')

In [21]:
#from sqlalchemy import inspect
#inspector = inspect(engine)
#print(inspector.get_table_names())
df.columns

Index(['order_id', 'order_date', 'ship_mode', 'segment', 'country', 'city',
       'state', 'postal_code', 'region', 'category', 'sub_category',
       'product_id', 'quantity', 'discount', 'sale_price', 'profit'],
      dtype='object')